# AlphaZero Chess — Google Colab Training

**Prerequisites:**
- Runtime → Change runtime type → **T4 GPU** (or better)
- Have your GitHub repo URL ready

**Workflow:**
1. Run cells 1–4 **once** per session to set up the environment
2. Run cell 5 to start or resume training
3. If the session disconnects, re-run cells 1 + 3 + 5 (Drive stays mounted between reconnects)
4. Run cell 6 at any time to download `best.pth.tar` locally

## Cell 1 — Clone repo & install dependencies
*(Run once per session)*

In [ ]:
# ── Edit this URL to point to your GitHub repo ────────────────────────────
REPO_URL = 'https://github.com/minghanminghan/chess-bot.git'
REPO_DIR = 'chess-bot'   # local clone directory name
# ─────────────────────────────────────────────────────────────────────────

import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'{REPO_DIR}/ already exists — skipping clone')

%cd {REPO_DIR}

# Install PyTorch with CUDA 12.1 (matches Colab's current CUDA)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
# Other deps
!pip install python-chess numpy tqdm -q

print('\nSetup complete.')

## Cell 2 — Mount Google Drive
*(Run once per session — checkpoints survive disconnects here)*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = '/content/drive/MyDrive/chess-bot-checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# Persist torch.compile kernel cache to Drive so restarts skip the ~60s warmup
TORCH_CACHE_DIR = CKPT_DIR + '/.torch_compile_cache'
os.makedirs(TORCH_CACHE_DIR, exist_ok=True)
os.environ['TORCHINDUCTOR_CACHE_DIR'] = TORCH_CACHE_DIR

print(f'Checkpoint dir  : {CKPT_DIR}')
print(f'torch.compile cache: {TORCH_CACHE_DIR}')
print(f'Existing checkpoints: {[f for f in os.listdir(CKPT_DIR) if f.endswith(".pth.tar")] or "(none)"}')

## Cell 3 — Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'No GPU detected. Go to Runtime → Change runtime type → Hardware accelerator → GPU'
)
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}')
print(f'RAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 4 — Pull latest code from GitHub
*(Run whenever you push local changes)*

In [ ]:
!git pull origin master

## Cell 5 — Run training
*(Re-run this cell after any session disconnect — set `RESUME = True` after the first run)*

In [ ]:
RESUME = True   # False = fresh run;  True = continue from best.pth.tar

resume_flag = '--resume' if RESUME else ''

!python train.py \
  --checkpoint-dir "{CKPT_DIR}" \
  {resume_flag}

## Cell 6 — Download best checkpoint to local machine

In [ ]:
import os
from google.colab import files

ckpt_path = os.path.join(CKPT_DIR, 'best.pth.tar')
if os.path.isfile(ckpt_path):
    files.download(ckpt_path)
    print('Download started.')
else:
    print(f'No checkpoint found at {ckpt_path}. Train first.')

## Cell 7 — Bottleneck profiler
Runs a short MCTS episode (50 moves × 50 sims) and reports exactly where wall time goes.
Run this before and after any optimisation to quantify the impact.

In [ ]:
import cProfile, pstats, io, time, sys, os
import numpy as np
import torch

sys.path.insert(0, '/content/chess-bot')
from chessbot.ChessGame import ChessGame
from chessbot.ChessNNet import ChessNNet
from MCTS import MCTS
from utils import dotdict

PROFILE_SIMS  = 50   # MCTS sims per move
PROFILE_MOVES = 50   # moves to profile (caps episode length)
NET_CH, NET_BL = 64, 4  # tiny network so compile is fast

cfg = dotdict(dict(numMCTSSims=PROFILE_SIMS, cpuct=1.0,
                   dirichlet_alpha=0.3, dirichlet_eps=0.25,
                   tempThreshold=30, num_channels=NET_CH, num_res_blocks=NET_BL))
game = ChessGame()
nnet = ChessNNet(game, cfg)   # torch.compile runs here on first forward pass

# ── 1. Manual hot-path timing ──────────────────────────────────────────────────
board     = game.getInitBoard()
canonical = game.getCanonicalForm(board, 1)
tensor    = canonical.to_tensor(canonical=True)
valids    = game.getValidMoves(canonical, 1)

N = 200
t = time.perf_counter()
for _ in range(N): nnet.predict(tensor)
t_predict = (time.perf_counter() - t) / N * 1000

t = time.perf_counter()
for _ in range(N): game.getValidMoves(canonical, 1)
t_valids = (time.perf_counter() - t) / N * 1000

t = time.perf_counter()
for _ in range(N): game.stringRepresentation(canonical)
t_strrepr = (time.perf_counter() - t) / N * 1000

t = time.perf_counter()
for _ in range(N): game.getNextState(board, 1, 4)   # e2e4
t_nexts = (time.perf_counter() - t) / N * 1000

t = time.perf_counter()
for _ in range(N): game.getGameEnded(board, 1)
t_ended = (time.perf_counter() - t) / N * 1000

# ── 2. cProfile over a full short episode ────────────────────────────────────
mcts  = MCTS(game, nnet, cfg)
board = game.getInitBoard()
cur   = 1

pr = cProfile.Profile()
pr.enable()
t0 = time.perf_counter()

moves = 0
while True:
    canonical = game.getCanonicalForm(board, cur)
    pi        = mcts.getActionProb(canonical, temp=1)
    action    = np.random.choice(len(pi), p=pi)
    board, cur = game.getNextState(board, cur, action)
    moves += 1
    if game.getGameEnded(board, cur) != 0 or moves >= PROFILE_MOVES:
        break

elapsed = time.perf_counter() - t0
pr.disable()

total_sims   = moves * PROFILE_SIMS
ms_per_move  = elapsed / moves * 1000
ms_per_sim   = elapsed / total_sims * 1000

# ── 3. Report ─────────────────────────────────────────────────────────────────
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f"\n{'='*62}")
print(f"  GPU: {gpu_name}   net: {NET_BL}×{NET_CH}ch   "
      f"{moves} moves × {PROFILE_SIMS} sims = {total_sims:,} total")
print(f"{'='*62}")
print(f"  Wall time : {elapsed:.1f}s  →  {ms_per_move:.0f} ms/move  |  {ms_per_sim:.2f} ms/sim")

print(f"\n  ── Per-call hot-path costs (avg of {N} calls) ──────────────")
rows = [
    ("nnet.predict()",         t_predict),
    ("getValidMoves()",        t_valids),
    ("stringRepresentation()", t_strrepr),
    ("getNextState()",         t_nexts),
    ("getGameEnded()",         t_ended),
]
total_hot = sum(c for _, c in rows)
for name, cost in rows:
    bar = '█' * int(cost / ms_per_sim * 30)
    print(f"  {name:27s}  {cost:7.2f} ms/call  {bar}")
print(f"  {'sum of above':27s}  {total_hot:7.2f} ms  ({total_hot/ms_per_sim*100:.0f}% of ms/sim)")
print(f"  {'MCTS tree overhead (rest)':27s}  {max(0, ms_per_sim - total_hot):7.2f} ms")

print(f"\n  ── cProfile top-20 by cumulative time ──────────────────────")
s = io.StringIO()
pstats.Stats(pr, stream=s).sort_stats('cumulative').print_stats(20)
# Print only the table (skip the header lines)
lines = s.getvalue().splitlines()
for line in lines:
    if line.strip():
        print(' ', line)